<a href="https://colab.research.google.com/github/Travis-Bickle10/bangalore-air-quality/blob/main/01_openaq_fetch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import time
import requests
import pandas as pd
from google.colab import userdata

key = userdata.get("OPENAQ_API_KEY")
print("Ready. Key loaded:", bool(key))

Ready. Key loaded: True


In [2]:
from google.colab import userdata
import requests
import pandas as pd

key = userdata.get("OPENAQ_API_KEY")
print("Key loaded:", bool(key))
print("Pandas version:", pd.__version__)
print("Requests version:", requests.__version__)

Key loaded: True
Pandas version: 2.2.2
Requests version: 2.32.4


In [3]:
response = requests.get(
    "https://api.openaq.org/v3/locations",
    headers={"X-API-Key": key},
    params={
        "coordinates": "12.9716,77.5946",
        "radius": 25000,                  # max allowed is 25km
        "limit": 20
    }
)

data = response.json()
print(f"Status: {response.status_code}")
print(f"Total stations found: {data['meta']['found']}")

stations = []
for loc in data["results"]:
    stations.append({
        "id":         loc["id"],
        "name":       loc["name"],
        "country":    loc["country"]["code"],
        "parameters": [s["parameter"]["name"] for s in loc["sensors"]],
        "from":       loc["datetimeFirst"]["local"][:10] if loc["datetimeFirst"] else "N/A",
        "to":         loc["datetimeLast"]["local"][:10] if loc["datetimeLast"] else "N/A",
        "latitude":   loc["coordinates"]["latitude"],
        "longitude":  loc["coordinates"]["longitude"],
    })

df_stations = pd.DataFrame(stations)
df_stations

Status: 200
Total stations found: >20


,id,name,country,parameters,from,to,latitude,longitude
0,412,"Peenya, Bengaluru - KSPCB",IN,"[co, no2, pm25, so2]",2016-03-22,2018-02-22,13.033900,77.513211
1,594,"BTM Layout, Bengaluru - KSPCB",IN,"[co, no2, o3, pm25, so2]",2016-03-22,2018-02-22,12.912811,77.609219
2,797,City Railway Station - KSPCB,IN,"[co, no2, pm10, so2]",2016-03-21,2018-02-22,12.977347,77.570697
3,2589,SaneguravaHalli - KSPCB,IN,"[co, no2, pm10, so2]",2016-03-22,2018-02-22,12.991669,77.545831
4,2592,"BWSSB Kadabesanahalli, Bengaluru - KSPCB",IN,"[co, no2, o3, pm25, so2]",2016-03-22,2018-02-22,12.938906,77.697272
5,5547,"BWSSB Kadabesanahalli, Bengaluru - CPCB",IN,"[co, no2, o3, pm10, pm25, so2]",2018-03-09,2022-08-02,12.935205,77.681449
6,5548,"BTM Layout, Bengaluru - CPCB",IN,"[co, co, no, no2, no2, nox, o3, o3, pm10, pm10...",2018-03-09,2026-03-30,12.913522,77.595080
7,5574,"City Railway Station, Bengaluru - KSPCB",IN,"[co, co, no, no2, no2, nox, pm10, pm10, so2, so2]",2018-03-09,2026-03-28,12.975684,77.566075
8,5607,"Peenya, Bengaluru - CPCB",IN,"[co, co, no, no2, no2, nox, o3, o3, pm10, pm10...",2018-03-09,2026-03-30,13.027020,77.494094
9,5644,"Sanegurava Halli, Bengaluru - KSPCB",IN,"[co, co, no, no2, no2, nox, pm10, pm10, relati...",2018-03-09,2026-03-30,12.990328,77.543138


In [4]:
# Filter to Indian stations only with relevant pollutants
target_pollutants = {"pm25", "pm10", "no2", "so2", "o3", "co"}

df_india = df_stations[df_stations["country"] == "IN"].copy()
df_india["useful_params"] = df_india["parameters"].apply(
    lambda p: [x for x in p if x in target_pollutants]
)
df_india = df_india[df_india["useful_params"].str.len() > 0]

print(f"Indian stations with relevant pollutants: {len(df_india)}")
df_india[["id", "name", "useful_params", "from", "to"]]

Indian stations with relevant pollutants: 20


,id,name,useful_params,from,to
0,412,"Peenya, Bengaluru - KSPCB","[co, no2, pm25, so2]",2016-03-22,2018-02-22
1,594,"BTM Layout, Bengaluru - KSPCB","[co, no2, o3, pm25, so2]",2016-03-22,2018-02-22
2,797,City Railway Station - KSPCB,"[co, no2, pm10, so2]",2016-03-21,2018-02-22
3,2589,SaneguravaHalli - KSPCB,"[co, no2, pm10, so2]",2016-03-22,2018-02-22
4,2592,"BWSSB Kadabesanahalli, Bengaluru - KSPCB","[co, no2, o3, pm25, so2]",2016-03-22,2018-02-22
5,5547,"BWSSB Kadabesanahalli, Bengaluru - CPCB","[co, no2, o3, pm10, pm25, so2]",2018-03-09,2022-08-02
6,5548,"BTM Layout, Bengaluru - CPCB","[co, co, no2, no2, o3, o3, pm10, pm10, pm25, p...",2018-03-09,2026-03-30
7,5574,"City Railway Station, Bengaluru - KSPCB","[co, co, no2, no2, pm10, pm10, so2, so2]",2018-03-09,2026-03-28
8,5607,"Peenya, Bengaluru - CPCB","[co, co, no2, no2, o3, o3, pm10, pm10, pm25, p...",2018-03-09,2026-03-30
9,5644,"Sanegurava Halli, Bengaluru - KSPCB","[co, co, no2, no2, pm10, pm10, so2, so2]",2018-03-09,2026-03-30


In [ ]:
# See full table without truncation
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 20)

print(df_india[["id", "name", "useful_params", "from", "to"]].to_string(index=False))

     id                                     name                                                useful_params       from         to
    412                Peenya, Bengaluru - KSPCB                                         [co, no2, pm25, so2] 2016-03-22 2018-02-22
    594            BTM Layout, Bengaluru - KSPCB                                     [co, no2, o3, pm25, so2] 2016-03-22 2018-02-22
    797             City Railway Station - KSPCB                                         [co, no2, pm10, so2] 2016-03-21 2018-02-22
   2589                  SaneguravaHalli - KSPCB                                         [co, no2, pm10, so2] 2016-03-22 2018-02-22
   2592 BWSSB Kadabesanahalli, Bengaluru - KSPCB                                     [co, no2, o3, pm25, so2] 2016-03-22 2018-02-22
   5547  BWSSB Kadabesanahalli, Bengaluru - CPCB                               [co, no2, o3, pm10, pm25, so2] 2018-03-09 2022-08-02
   5548             BTM Layout, Bengaluru - CPCB [co, co, no2, no2, o3, o3, 

In [7]:
import time
import json
from datetime import datetime

TARGET_PARAMS = {"pm25", "pm10", "no2", "so2", "o3", "co"}

STATIONS = {
    5548:  "BTM Layout",
    5574:  "City Railway Station",
    5607:  "Peenya",
    6973:  "Jayanagar",
    6974:  "Bapuji Nagar",
    6975:  "Silk Board",
    6983:  "Hombegowda Nagar",
    6984:  "Hebbal",
}

all_readings = []

for station_id, station_name in STATIONS.items():
    print(f"\nFetching: {station_name} (id={station_id})")
    page = 1
    station_total = 0

    while True:
        response = requests.get(
            f"https://api.openaq.org/v3/locations/{station_id}/sensors",
            headers={"X-API-Key": key},
        )

        # Get sensor IDs for this station
        sensors = response.json().get("results", [])
        sensor_ids = [
            s["id"] for s in sensors
            if s["parameter"]["name"] in TARGET_PARAMS
        ]
        break  # just need sensor IDs once

    for sensor_id in sensor_ids:
        page = 1
        while True:
            r = requests.get(
                f"https://api.openaq.org/v3/sensors/{sensor_id}/measurements/hourly",
                headers={"X-API-Key": key},
                params={
                    "date_from": "2018-01-01",
                    "date_to":   "2024-12-31",
                    "limit":     1000,
                    "page":      page,
                }
            )

            if r.status_code != 200:
                print(f"  Error {r.status_code} on sensor {sensor_id}, page {page}")
                break

            results = r.json().get("results", [])
            if not results:
                break

            for reading in results:
                all_readings.append({
                    "station_id":   station_id,
                    "station_name": station_name,
                    "sensor_id":    sensor_id,
                    "parameter":    reading["parameter"]["name"],
                    "value":        reading["value"],
                    "date":         reading["period"]["datetimeFrom"]["local"][:10],
                })

            station_total += len(results)
            print(f"  sensor {sensor_id} · page {page} · {station_total} rows so far", end="\r")
            page += 1
            time.sleep(0.5)

print(f"\n\nDone. Total raw readings: {len(all_readings):,}")
df_raw = pd.DataFrame(all_readings)
df_raw.head()


Fetching: BTM Layout (id=5548)
  Error 500 on sensor 14657, page 1

Fetching: City Railway Station (id=5574)

Fetching: Peenya (id=5607)
  Error 500 on sensor 12235372, page 3

Fetching: Jayanagar (id=6973)

Fetching: Bapuji Nagar (id=6974)

Fetching: Silk Board (id=6975)

Fetching: Hombegowda Nagar (id=6983)
  Error 500 on sensor 12235260, page 3

Fetching: Hebbal (id=6984)


Done. Total raw readings: 951,628


,station_id,station_name,sensor_id,parameter,value,date
0,5548,BTM Layout,14639,no2,56.0,2018-03-09
1,5548,BTM Layout,14639,no2,56.0,2018-03-09
2,5548,BTM Layout,14639,no2,54.0,2018-03-10
3,5548,BTM Layout,14639,no2,48.0,2018-03-10
4,5548,BTM Layout,14639,no2,42.0,2018-03-10


In [8]:
df_raw.to_csv("df_raw_openaq.csv", index=False)
print("Saved!")

Saved!


In [9]:
# ── Step 1: Remove bad values ─────────────────────────────────────
df_clean = df_raw.copy()
df_clean = df_clean[df_clean["value"] >= 0]       # drop negatives
df_clean = df_clean[df_clean["value"] < 900]       # drop instrument errors
df_clean["date"] = pd.to_datetime(df_clean["date"])
df_clean["year"] = df_clean["date"].dt.year

print(f"Rows after cleaning: {len(df_clean):,}  (removed {len(df_raw)-len(df_clean):,})")

# ── Step 2: Average duplicate sensors per station/day ─────────────
# Some stations have two sensors for the same pollutant — average them
df_deduped = (
    df_clean
    .groupby(["station_id", "station_name", "parameter", "date", "year"])["value"]
    .mean()
    .reset_index()
)

# ── Step 3: Resample to annual means per station ──────────────────
df_annual_station = (
    df_deduped
    .groupby(["station_name", "parameter", "year"])
    .agg(
        mean_value = ("value", "mean"),
        coverage   = ("value", "count"),   # number of daily readings
    )
    .reset_index()
)

# Flag low coverage years (less than 146 days = under 40% of year)
df_annual_station["reliable"] = df_annual_station["coverage"] >= 146
print(f"\nUnreliable station-years (low coverage):")
print(df_annual_station[~df_annual_station["reliable"]][["station_name","parameter","year","coverage"]])

# ── Step 4: Average across all stations → city-level annual mean ──
df_city = (
    df_annual_station[df_annual_station["reliable"]]
    .groupby(["parameter", "year"])
    .agg(city_mean = ("mean_value", "mean"))
    .reset_index()
)

# ── Step 5: Pivot to wide format — one row per year ───────────────
df_final = df_city.pivot(index="year", columns="parameter", values="city_mean").round(2)
df_final.columns.name = None
df_final = df_final.reset_index()

print(f"\nFinal city-level annual means:")
df_final

Rows after cleaning: 926,036  (removed 25,592)

Unreliable station-years (low coverage):
    station_name parameter  year  coverage
1     BTM Layout        co  2026        66
2     BTM Layout       no2  2018       141
8     BTM Layout       no2  2026        66
9     BTM Layout        o3  2018       143
15    BTM Layout        o3  2026        66
..           ...       ...   ...       ...
292   Silk Board      pm10  2026        77
293   Silk Board      pm25  2018        97
299   Silk Board      pm25  2026        77
300   Silk Board       so2  2018        95
306   Silk Board       so2  2026        77

[96 rows x 4 columns]

Final city-level annual means:


,year,co,no2,o3,pm10,pm25,so2
0,2018,NaN,19.98,NaN,NaN,47.60,9.08
1,2019,296.62,24.24,31.18,56.66,24.62,5.86
2,2020,351.42,20.24,31.95,65.37,27.99,7.06
3,2021,269.07,25.93,28.74,71.42,36.05,6.28
4,2022,239.68,19.28,22.76,71.46,31.63,7.59
5,2025,0.62,18.56,24.19,72.27,24.54,9.45


In [10]:
# Save raw fetch
df_raw.to_csv("df_raw_openaq.csv", index=False)

# Save final clean annual means
df_final.to_csv("openaq_bangalore_annual.csv", index=False)

# Save to Google Drive for persistence
from google.colab import drive
drive.mount("/content/drive")

import os
os.makedirs("/content/drive/MyDrive/bangalore_air_quality", exist_ok=True)

df_raw.to_csv("/content/drive/MyDrive/bangalore_air_quality/df_raw_openaq.csv", index=False)
df_final.to_csv("/content/drive/MyDrive/bangalore_air_quality/openaq_bangalore_annual.csv", index=False)

print("Saved to Drive successfully")
print(df_final)

Mounted at /content/drive
Saved to Drive successfully
   year      co    no2     o3   pm10   pm25   so2
0  2018     NaN  19.98    NaN    NaN  47.60  9.08
1  2019  296.62  24.24  31.18  56.66  24.62  5.86
2  2020  351.42  20.24  31.95  65.37  27.99  7.06
3  2021  269.07  25.93  28.74  71.42  36.05  6.28
4  2022  239.68  19.28  22.76  71.46  31.63  7.59
5  2025    0.62  18.56  24.19  72.27  24.54  9.45


In [11]:
# Check what years exist in the raw data
print("Years in raw data:")
print(df_raw["date"].str[:4].value_counts().sort_index())

Years in raw data:
date
2018     79401
2019    145074
2020    163241
2021    103237
2022    123123
2025    271428
2026     66124
Name: count, dtype: int64


In [12]:
# ── Fix 1: Drop 2025 CO — sensor anomaly ─────────────────────────
df_final.loc[df_final["year"] == 2025, "co"] = None

# ── Fix 2: Drop 2026 — partial year, unreliable ──────────────────
df_final = df_final[df_final["year"] != 2026]

# ── Fix 3: Mark 2023 and 2024 as missing explicitly ──────────────
missing_years = pd.DataFrame({"year": [2023, 2024]})
df_final = pd.concat([df_final, missing_years], ignore_index=True)
df_final = df_final.sort_values("year").reset_index(drop=True)

# ── Summary ───────────────────────────────────────────────────────
print("Final cleaned dataset:")
print(df_final.to_string(index=False))

print("\nCoverage summary:")
for col in ["pm25", "pm10", "no2", "so2", "o3", "co"]:
    valid = df_final[col].notna().sum()
    print(f"  {col}: {valid} years of data")

Final cleaned dataset:
 year     co   no2    o3  pm10  pm25  so2
 2018    NaN 19.98   NaN   NaN 47.60 9.08
 2019 296.62 24.24 31.18 56.66 24.62 5.86
 2020 351.42 20.24 31.95 65.37 27.99 7.06
 2021 269.07 25.93 28.74 71.42 36.05 6.28
 2022 239.68 19.28 22.76 71.46 31.63 7.59
 2023    NaN   NaN   NaN   NaN   NaN  NaN
 2024    NaN   NaN   NaN   NaN   NaN  NaN
 2025    NaN 18.56 24.19 72.27 24.54 9.45

Coverage summary:
  pm25: 6 years of data
  pm10: 5 years of data
  no2: 6 years of data
  so2: 6 years of data
  o3: 5 years of data
  co: 4 years of data


In [13]:
# Save final cleaned OpenAQ data
df_final.to_csv("openaq_bangalore_annual.csv", index=False)
df_final.to_csv("/content/drive/MyDrive/bangalore_air_quality/openaq_bangalore_annual.csv", index=False)

print("OpenAQ data saved — years covered:")
print(df_final[df_final.notna().any(axis=1)]["year"].values)

OpenAQ data saved — years covered:
[2018 2019 2020 2021 2022 2023 2024 2025]
